# Dusha Smoke Test — 1 batch, 1 epoch

Проверяет что весь пайплайн работает: датасет, модель, тренер, чекпоинт, матрица.  
Обучается на одном батче, одна агрегация (`majority`).  
Запускай перед полным `dusha_train_kaggle.ipynb`.

## 1. GPU check

In [ ]:
import subprocess, sys, os
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available")

## 2. Install dependencies

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torchaudio', 'transformers>=4.40', 'datasets>=2.18', 'peft>=0.10',
    'scikit-learn', 'matplotlib', 'seaborn', 'soundfile', 'pyyaml', 'tqdm',
], check=True)
print('Done.')

## 3. Clone repo

In [ ]:
REPO_URL = 'https://github.com/aibryanov/speech_emo_finetune.git'
REPO_DIR = 'speech_emo_finetune'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())

## 4. Config

In [ ]:
import warnings, logging, pathlib, glob, pandas as pd
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)

from src.config import load_config

# ── укажи имена датасетов ─────────────────────────────────────────────────────
AGGREGATED_DATASET = '<твой-датасет-train>'
TEST_DATASET       = '<твой-датасет-test>'
# ─────────────────────────────────────────────────────────────────────────────

AGG_ROOT  = pathlib.Path(f'/kaggle/input/{AGGREGATED_DATASET}')
TEST_ROOT = pathlib.Path(f'/kaggle/input/{TEST_DATASET}')

SMOKE_TRAIN_TSV = AGG_ROOT  / 'aggregated_majority.tsv'
SMOKE_TEST_TSV  = TEST_ROOT / 'aggregated_majority.tsv'

# автодетект через glob
_found = glob.glob('/kaggle/input/**/crowd_train/wavs', recursive=True)
if not _found:
    raise RuntimeError('crowd_train/wavs not found — проверь что датасет добавлен в Kaggle input')
_base = str(pathlib.Path(_found[0]).parent.parent)
TRAIN_AUDIO_DIR = f'{_base}/crowd_train'
TEST_AUDIO_DIR  = f'{_base}/crowd_test'

print(f'Dataset base    : {_base}')
print(f'TRAIN_AUDIO_DIR : {TRAIN_AUDIO_DIR}  exists={pathlib.Path(TRAIN_AUDIO_DIR).exists()}')
print(f'TEST_AUDIO_DIR  : {TEST_AUDIO_DIR}   exists={pathlib.Path(TEST_AUDIO_DIR).exists()}')
print(f'train TSV : {SMOKE_TRAIN_TSV}  exists={SMOKE_TRAIN_TSV.exists()}')
print(f'test  TSV : {SMOKE_TEST_TSV}   exists={SMOKE_TEST_TSV.exists()}')

if SMOKE_TRAIN_TSV.exists():
    df = pd.read_csv(SMOKE_TRAIN_TSV, sep='\t', nrows=3)
    print('\nTrain TSV preview:')
    print(df[['hash_id', 'audio_path', 'aggregated_emo']].to_string(index=False))
    first = pathlib.Path(TRAIN_AUDIO_DIR) / df['audio_path'].iloc[0]
    print(f'First audio: {first}  exists={first.exists()}')

## 5. Smoke test — 1 batch training

In [ ]:
import gc, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Subset, DataLoader
from functools import partial
from transformers import AutoFeatureExtractor

from src.config import load_config
from src.dataset import get_dusha_dataloaders, get_dusha_test_dataloader, collate_fn
from src.models import build_model
from src.trainer import Trainer

gc.collect()
torch.cuda.empty_cache()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = load_config('configs/wavlm_base_dusha_full.yaml')
config.aggregated_tsv      = str(SMOKE_TRAIN_TSV)
config.audio_dir           = TRAIN_AUDIO_DIR
config.num_workers         = 0
config.epochs              = 1
config.batch_size          = 4
config.save_every_n_epochs = 0
config.run_name            = 'smoke_test'
config.output_dir          = 'outputs/smoke_test'

processor = AutoFeatureExtractor.from_pretrained(
    config.processor_name or config.model_name)

train_loader_full, dev_loader_full, _ = get_dusha_dataloaders(config, processor)

BATCH = config.batch_size
max_len = int(config.max_audio_len_s * 16_000)
_collate = partial(collate_fn, max_len_samples=max_len)

train_subset = Subset(train_loader_full.dataset, list(range(min(BATCH, len(train_loader_full.dataset)))))
dev_subset   = Subset(dev_loader_full.dataset,   list(range(min(BATCH, len(dev_loader_full.dataset)))))

train_loader = DataLoader(train_subset, batch_size=BATCH, shuffle=False, collate_fn=_collate)
dev_loader   = DataLoader(dev_subset,   batch_size=BATCH, shuffle=False, collate_fn=_collate)

print(f'Smoke train: {len(train_subset)} samples', flush=True)
print(f'Smoke dev:   {len(dev_subset)} samples', flush=True)

model = build_model(config)
if torch.cuda.device_count() >= 2:
    model = nn.DataParallel(model)
    print(f'DataParallel: {torch.cuda.device_count()} GPUs', flush=True)

trainer = Trainer(model, config, train_loader, dev_loader, dev_loader, device)
trainer.fit()
print('\n✓ Training OK')

## 6. Проверка наличия аудиофайлов в test TSV

In [ ]:
df_test = pd.read_csv(SMOKE_TEST_TSV, sep='\t')
missing = []
for row in df_test.itertuples():
    p = pathlib.Path(TEST_AUDIO_DIR) / row.audio_path
    if not p.exists():
        missing.append((row.hash_id, row.audio_path))

print(f'Test TSV rows  : {len(df_test):,}')
print(f'Missing in test: {len(missing):,}')
if missing:
    print('\nFirst 10 missing:')
    for hid, ap in missing[:10]:
        # проверим — может файл лежит в train?
        in_train = (pathlib.Path(TRAIN_AUDIO_DIR) / ap).exists()
        print(f'  {ap}  in_train={in_train}')
    if len(missing) == len(df_test):
        print('\n⚠ Все файлы отсутствуют в crowd_test — возможно TSV сгенерирован из train-данных!')
else:
    print('✓ Все аудиофайлы найдены в crowd_test')

In [ ]:
from sklearn.metrics import balanced_accuracy_score
from tqdm.auto import tqdm

ckpt_path = pathlib.Path(config.output_dir) / 'best_model.pt'
print(f'Checkpoint: {ckpt_path}  exists={ckpt_path.exists()}')

model2 = build_model(config)
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
model2.load_state_dict(ckpt['model_state_dict'])
model2.eval().to(device)
print('✓ Checkpoint loaded')

test_loader_full = get_dusha_test_dataloader(
    str(SMOKE_TEST_TSV), TEST_AUDIO_DIR, processor, config)

test_subset = Subset(
    test_loader_full.dataset,
    list(range(min(BATCH, len(test_loader_full.dataset))))
)
test_loader = DataLoader(test_subset, batch_size=BATCH, shuffle=False, collate_fn=_collate)

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test inference', leave=False):
        inputs = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
        preds = model2(**inputs).argmax(dim=-1).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(batch['labels'].numpy())

wacc = balanced_accuracy_score(
    np.concatenate(all_labels), np.concatenate(all_preds))
print(f'\n✓ Inference OK  |  weighted_accuracy (smoke): {wacc:.4f}')
print('\nAll smoke tests passed — ready to run dusha_train_kaggle.ipynb')